In [1]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json
from pathlib import Path

In [2]:
input_data = "input_files/relationship_extraction_0081.csv"
output_data = "output_files/relationship_extraction_0081.xlsx"

In [3]:
from dotenv import load_dotenv

env_path = Path('.') / '.env'
load_dotenv(dotenv_path=env_path)## Open AI API

True

## common functions

In [4]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [5]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [6]:
"""
    This function is different a bit among different tasks, here is for the relationship extraction task
"""

def construct_annotation(df, results, task_name="semantic_matching",rule_id="0081"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
        the options of rule_id are 0081, 0001, and 0145
            if rule_id is 0081, relation_type is Reversal
            if rule_id is 0001, relation_type is CombinationErr
            if rule_id is 0145, relation_type is Inappropriateness
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []

    if rule_id == "0081":
        relation_type = "Reversal"
    elif rule_id == "0001":
        relation_type = "CombinationErr"
    else:
        relation_type = "Inappropriateness"

    
    for res in results:
        segmentation_name = res.get("segmentation_name")
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)
        potential_seg_names = []

        ## 
        for seg_name in real_segmentation_names:
            normalized_segmentation_name = segmentation_name.lower()
            normalized_seg_name = seg_name.lower()
            if (normalized_segmentation_name in normalized_seg_name) or (normalized_seg_name in normalized_segmentation_name):
                potential_seg_names.append(seg_name)
        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        
        answers = [(tag.get("tag1"),tag.get("tag2")) for tag in res.get("error_tags")] ##special for relationship extraction task
        flattened_answers = [x for pair in answers for x in pair] ##special for relationship extraction task, which is easier to check if the tag is in the specific document

        # convert the answers to triplet format
        answers = [(pair[0],relation_type,pair[1]) for pair in answers]
        for segmentation_name_ in potential_seg_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True

            for ans in flattened_answers:
                ## check the calculation linkbase
                if ans in temp_cal:
                    continue
                elif ans.replace(":","") in temp_cal:
                    continue
                elif ans.replace(":","_") in temp_cal:
                    continue
                else:
                    is_appear = False
                    break
           
            if is_appear:
                parse = True
                cal = temp_cal
                pre = temp_pre
                defn = temp_defn
                lab = temp_lab
                ins = temp_ins
                sch = temp_sch
                seg_path = os.path.join(folder_name, segmentation_name_)
                break
        
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(None)
        outputByLLM.append(res)
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for relationship extraction task

In [7]:
re_81_df = pd.read_csv(input_data)

In [8]:
re_81_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0081.9278,"2022-01-03 07:00:36,513 [DQC.US.0081.9278] The...",2022-01-03,ERROR,DQC,81,1800373,"GOLDENWELL BIOTECH, INC.",2022-01-03 06:29:00,0001477932-22-000010,US GAAP 2020,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1800373...,2022-01-03,2022,409763,https://www.sec.gov/Archives/edgar/data/180037...,2000,10q-gdwl-20210930
1,DQC.US.0081.9277,"2022-01-14 12:30:34,917 [DQC.US.0081.9277] The...",2022-01-14,ERROR,DQC,81,1350156,PREAXIA HEALTH CARE PAYMENT SYSTEMS INC.,2022-01-14 12:02:00,0001262463-22-000015,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_058_edei%2D%2DD...,10-Q,http://www.sec.gov/Archives/edgar/data/1350156...,2022-01-14,2022,412124,https://www.sec.gov/Archives/edgar/data/135015...,5700,10q-paxh-20211130
2,DQC.US.0081.9278,"2022-01-14 12:30:34,917 [DQC.US.0081.9278] The...",2022-01-14,ERROR,DQC,81,1350156,PREAXIA HEALTH CARE PAYMENT SYSTEMS INC.,2022-01-14 12:02:00,0001262463-22-000015,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_058_edei%2D%2DD...,10-Q,http://www.sec.gov/Archives/edgar/data/1350156...,2022-01-14,2022,412124,https://www.sec.gov/Archives/edgar/data/135015...,5700,10q-paxh-20211130
3,DQC.US.0081.9278,"2022-01-14 12:30:34,917 [DQC.US.0081.9278] The...",2022-01-14,ERROR,DQC,81,1350156,PREAXIA HEALTH CARE PAYMENT SYSTEMS INC.,2022-01-14 12:02:00,0001262463-22-000015,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_058_edei%2D%2DD...,10-Q,http://www.sec.gov/Archives/edgar/data/1350156...,2022-01-14,2022,412124,https://www.sec.gov/Archives/edgar/data/135015...,5700,10q-paxh-20211130
4,DQC.US.0081.9278,"2022-03-09 20:02:26,631 [DQC.US.0081.9278] The...",2022-03-10,ERROR,DQC,81,1579010,"LIFE ON EARTH, INC.",2022-01-14 16:34:00,0001017386-22-000026,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_050_edei%2D%2DC...,10-Q,http://www.sec.gov/Archives/edgar/data/1579010...,2022-01-14,2022,412216,https://www.sec.gov/Archives/edgar/data/157901...,2080,10q-lfer-20211130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,DQC.US.0081.9278,"2024-11-18 06:45:54,595 [DQC.US.0081.9278] The...",2024-11-18,ERROR,DQC,81,1543652,Free Flow USA Inc.,2024-11-14 12:55:00,0001096906-24-002132,US GAAP 2024,Produced by gain loss using EDGARsuite softwar...,10-Q,http://www.sec.gov/Archives/edgar/data/1543652...,2024-11-14,2024,781181,https://www.sec.gov/Archives/edgar/data/154365...,5531,10q-fflo-20240930
102,DQC.US.0081.9277,"2024-11-18 06:27:53,326 [DQC.US.0081.9277] The...",2024-11-18,ERROR,DQC,81,1404804,OMNITEK ENGINEERING CORP.,2024-11-14 11:23:00,0001096906-24-002123,US GAAP 2024,Produced by gain loss using EDGARsuite softwar...,10-Q,http://www.sec.gov/Archives/edgar/data/1404804...,2024-11-14,2024,781133,https://www.sec.gov/Archives/edgar/data/140480...,3714,10q-omtk-20240930
103,DQC.US.0081.9277,"2024-11-18 06:27:53,327 [DQC.US.0081.9277] The...",2024-11-18,ERROR,DQC,81,1404804,OMNITEK ENGINEERING CORP.,2024-11-14 11:23:00,0001096906-24-002123,US GAAP 2024,Produced by gain loss using EDGARsuite softwar...,10-Q,http://www.sec.gov/Archives/edgar/data/1404804...,2024-11-14,2024,781133,https://www.sec.gov/Archives/edgar/data/140480...,3714,10q-omtk-20240930
104,DQC.US.0081.9277,"2024-11-18 06:27:53,327 [DQC.US.0081.9277] The...",2024-11-18,ERROR,DQC,81,1404804,OMNITEK ENGINEERING CORP.,2024-11-14 11:23:00,0001096906-24-002123,US GAAP 2024,Produced by gain loss using EDGARsuite softwar...,10-Q,http://www.sec.gov/Archives/edgar/data/1404804...,2024-11-14,2024,781133,https://www.sec.gov/Archives/edgar/data/140480...,3714,10q-omtk-20240930


## LLM judge assisstant

In [9]:
Client = OpenAI()

In [10]:
re_81_system_prompt = """You are given a DQC validation message. Your task is to extract two specific pieces of information:

1. **Statement Name**: Extract the name of the financial statement mentioned in the message. If it appears in a phrase like "00000002 - Statement - BALANCE SHEET", ignore any prefixes such as ID numbers or "- Statement -". Only return the clean name of the statement (e.g., "BALANCE SHEET").

2. **Erroneous us-gaap Tag Relationships**: From all `us-gaap:` tags mentioned in the message, identify pairs of tags where the relationship between them is described as incorrect or inconsistent. A tag pair should be extracted **only if** it satisfies both of the following conditions:
   - The two tags are stated to be in a specific structural relationship in the filer’s taxonomy (e.g., sibling, parent-child).
   - The message clearly indicates that this relationship contradicts the definition in the official US GAAP taxonomy, or requires reviewer attention for potential misclassification.

3. **Reasoning**: For each extracted tag pair, provide a short explanation summarizing the inconsistency or incorrect relationship between them as described in the message.

Output your result as a structured JSON in the following format:
```json
{
  "statement_name": "...",
  "error_tags": [
    {
      "tag1": "...",
      "tag2": "...",
      "reason": "..."
    }
  ]
}
"""

In [11]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": re_81_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [12]:
res = []
for _, row in tqdm(re_81_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## covert the extracted statement name in the message to PascalCase format as the segmentation name
    segmentation_name = to_pascal_case(response.get("statement_name"))

    ## record the folder name
    response["folder_name"] = folder_name
    ## record the extracted segmenataion name
    response["segmentation_name"] = segmentation_name
    res.append(response)

106it [02:36,  1.47s/it]


In [13]:
res[6]

{'statement_name': 'Condensed Consolidated Balance Sheets',
 'error_tags': [{'tag1': 'us-gaap:IntangibleAssetsCurrent',
   'tag2': 'us-gaap:AssetsCurrent',
   'reason': 'In the filer’s taxonomy, us-gaap:IntangibleAssetsCurrent is represented as a sibling of us-gaap:AssetsCurrent, but according to the official US GAAP taxonomy, it should be a descendant (child) of us-gaap:AssetsCurrent, indicating an incorrect structural relationship.'}],
 'folder_name': '10q-lfer-20211130',
 'segmentation_name': 'CondensedConsolidatedBalanceSheets'}

In [14]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    the options of rule_id are 0081, 0001, and 0145
"""
new_re_81_df = construct_annotation(re_81_df, res, task_name="relationship_extraction", rule_id="0081")

In [15]:
new_re_81_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0081.9278,"2022-01-03 07:00:36,513 [DQC.US.0081.9278] The...",2022-01-03,ERROR,DQC,81,1800373,"GOLDENWELL BIOTECH, INC.",2022-01-03 06:29:00,0001477932-22-000010,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:ProceedsFromIssuanceOfCommonStock, R...",10q-gdwl-20210930/StatementOfCashFlowsUnaudited,True,{'statement_name': 'STATEMENT OF CASH FLOWS (U...
1,DQC.US.0081.9277,"2022-01-14 12:30:34,917 [DQC.US.0081.9277] The...",2022-01-14,ERROR,DQC,81,1350156,PREAXIA HEALTH CARE PAYMENT SYSTEMS INC.,2022-01-14 12:02:00,0001262463-22-000015,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:ProceedsFromLoans, Reversal, us-gaap...",10q-paxh-20211130/CondensedConsolidatedStateme...,True,{'statement_name': 'CONDENSED CONSOLIDATED STA...
2,DQC.US.0081.9278,"2022-01-14 12:30:34,917 [DQC.US.0081.9278] The...",2022-01-14,ERROR,DQC,81,1350156,PREAXIA HEALTH CARE PAYMENT SYSTEMS INC.,2022-01-14 12:02:00,0001262463-22-000015,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:AccountsPayableRelatedPartiesCurrent...,10q-paxh-20211130/CondensedConsolidatedBalance...,True,{'statement_name': 'CONDENSED CONSOLIDATED BAL...
3,DQC.US.0081.9278,"2022-01-14 12:30:34,917 [DQC.US.0081.9278] The...",2022-01-14,ERROR,DQC,81,1350156,PREAXIA HEALTH CARE PAYMENT SYSTEMS INC.,2022-01-14 12:02:00,0001262463-22-000015,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:AccountsPayableRelatedPartiesCurrent...,10q-paxh-20211130/CondensedConsolidatedBalance...,True,{'statement_name': 'CONDENSED CONSOLIDATED BAL...
4,DQC.US.0081.9278,"2022-03-09 20:02:26,631 [DQC.US.0081.9278] The...",2022-03-10,ERROR,DQC,81,1579010,"LIFE ON EARTH, INC.",2022-01-14 16:34:00,0001017386-22-000026,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:ConvertibleNotesPayable, Reversal, u...",10q-lfer-20211130/BalanceSheets,True,{'statement_name': 'Condensed Consolidated Bal...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,DQC.US.0081.9278,"2024-11-18 06:45:54,595 [DQC.US.0081.9278] The...",2024-11-18,ERROR,DQC,81,1543652,Free Flow USA Inc.,2024-11-14 12:55:00,0001096906-24-002132,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<linkb...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:IncreaseDecreaseInNotesPayableCurren...,10q-fflo-20240930/idr_StatementUnauditedConden...,True,{'statement_name': 'Unaudited Condensed Consol...
102,DQC.US.0081.9277,"2024-11-18 06:27:53,326 [DQC.US.0081.9277] The...",2024-11-18,ERROR,DQC,81,1404804,OMNITEK ENGINEERING CORP.,2024-11-14 11:23:00,0001096906-24-002123,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<linkb...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl 

In [16]:
new_re_81_df.to_excel(output_data,index=False)